# Notebook 2: Centralized MILP for BRP Portfolio Optimization

## References

1. **Pandžić, H., Morales, J. M., Conejo, A. J., Kuzle, I. (2013).** *"Offering model for a virtual power plant based on stochastic programming."* Applied Energy, 105, 282–292. [DOI: 10.1016/j.apenergy.2012.12.077](https://doi.org/10.1016/j.apenergy.2012.12.077)

2. **Burger, S., Chaves-Ávila, J. P., Batlle, C., Pérez-Arriaga, I. J. (2017).** *"A review of the value of aggregators in electricity systems."* Renewable and Sustainable Energy Reviews, 77, 395–405. [DOI: 10.1016/j.rser.2017.04.014](https://doi.org/10.1016/j.rser.2017.04.014)

3. **Ottesen, S. Ø., Tomasgard, A., Fleten, S.-E. (2018).** *"Multi market bidding strategies for demand side flexibility aggregators in electricity markets."* Energy, 149, 120–134. [DOI: 10.1016/j.energy.2018.01.187](https://doi.org/10.1016/j.energy.2018.01.187)

## What these papers bring

**Pandžić et al. (2013)** propose a VPP offering model where distributed energy resources (DERs) are aggregated and jointly scheduled to participate in day-ahead markets. The key insight is that aggregation creates **portfolio effects** — individual forecast errors partially cancel out, and flexible assets can compensate for each other.

**Burger et al. (2017)** provide an extensive review of aggregator business models, identifying the key value streams: energy arbitrage, portfolio balancing, and ancillary services. They highlight that the aggregator (or BRP) benefits from **internalized netting** of prosumer positions.

**Ottesen et al. (2018)** formulate a multi-market bidding strategy for a flexibility aggregator participating in day-ahead and balancing markets, showing that coordinated bidding across markets significantly outperforms sequential participation.

## What is implemented below

We extend the single-site MILP from Notebook 1 to a **centralized BRP model** with $N$ prosumer sites. The BRP:
1. Jointly optimizes all BESS units across the portfolio
2. Submits a single **day-ahead nomination** (aggregated net position) to the market
3. Faces **imbalance costs** for deviations from the nomination
4. Benefits from **internal netting** — surplus at one site offsets deficit at another

We compare the centralized portfolio cost against the sum of independently optimized single-site costs.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pulp

np.random.seed(42)


## 1. Generate Multi-Site Data

We create $N = 5$ prosumer sites with different PV sizes, load profiles, and battery capacities.
This represents a realistic small BRP portfolio in the Czech Republic.


In [ ]:
T = 24
hours = np.arange(T)
N_sites = 5

# Site parameters: (PV_peak_kW, Load_base_kW, E_max_kWh, P_max_kW)
site_params = [
    (5.0, 1.5, 10.0, 5.0),   # Site 0: residential with PV+BESS
    (8.0, 3.0, 15.0, 7.0),   # Site 1: larger residential
    (3.0, 2.0, 5.0, 3.0),    # Site 2: small residential
    (12.0, 5.0, 20.0, 10.0), # Site 3: small commercial
    (6.0, 2.5, 8.0, 4.0),    # Site 4: residential
]

# Generate profiles for each site
pv_profiles = []
load_profiles = []

for i, (pv_peak, load_base, _, _) in enumerate(site_params):
    pv_i = np.maximum(0, pv_peak * np.exp(-0.5 * ((hours - 12 + 0.5*(i-2)) / 3.0)**2)
                       + 0.2 * np.random.randn(T))
    pv_i = np.maximum(pv_i, 0)
    pv_profiles.append(pv_i)

    load_i = load_base + 0.8 * np.exp(-0.5 * ((hours - 7 - 0.5*i) / 2.0)**2) + \
             1.2 * np.exp(-0.5 * ((hours - 19 + 0.3*i) / 2.5)**2) + \
             0.3 * np.random.randn(T)
    load_i = np.maximum(load_i, 0.3)
    load_profiles.append(load_i)

# Day-ahead prices (EUR/MWh) — OTE CZ style
base_price = 40 + 20 * np.sin(2 * np.pi * (hours - 6) / 24) + \
             15 * np.exp(-0.5 * ((hours - 18) / 3.0)**2) + \
             5 * np.random.randn(T)
price_da = np.maximum(base_price, 10.0)

# Imbalance prices: dual-price approximation
# Surplus (BRP nominated more than needed): earns less than DA
# Deficit (BRP nominated less than needed): pays more than DA
price_imb_surplus = price_da * 0.7   # surplus returned at discount
price_imb_deficit = price_da * 1.3   # deficit purchased at premium

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for i in range(N_sites):
    ax = axes[i // 3, i % 3]
    ax.bar(hours, pv_profiles[i], color='gold', alpha=0.7, label='PV')
    ax.plot(hours, load_profiles[i], 'r-o', markersize=2, label='Load')
    ax.set_title(f'Site {i} (PV={site_params[i][0]}kW, E={site_params[i][2]}kWh)')
    ax.set_xlabel('Hour'); ax.legend(fontsize=7)

axes[1, 2].plot(hours, price_da, 'b-o', markersize=3, label='DA price')
axes[1, 2].plot(hours, price_imb_deficit, 'r--', alpha=0.6, label='Deficit penalty')
axes[1, 2].plot(hours, price_imb_surplus, 'g--', alpha=0.6, label='Surplus return')
axes[1, 2].set_title('Prices (EUR/MWh)'); axes[1, 2].legend(fontsize=7)
axes[1, 2].set_xlabel('Hour')

plt.tight_layout()
plt.savefig('/tmp/nb2_sites.png', dpi=100)
plt.show()
print(f"Generated {N_sites} sites with {T}-hour profiles.")


## 2. Independent Single-Site Optimization (Baseline)

First, we solve each site independently (the current company approach) and sum the costs.


In [ ]:
def solve_single_site(pv, load, price_buy, price_sell, E_max, E_min_frac, P_max,
                      eta_ch=0.95, eta_dis=0.95, SoC_init_frac=0.5):
    T = len(pv)
    E_min = E_max * E_min_frac
    SoC_init = E_max * SoC_init_frac
    dt = 1.0

    prob = pulp.LpProblem("SingleSite", pulp.LpMinimize)

    p_ch = [pulp.LpVariable(f"ch_{t}", 0, P_max) for t in range(T)]
    p_dis = [pulp.LpVariable(f"dis_{t}", 0, P_max) for t in range(T)]
    p_buy = [pulp.LpVariable(f"buy_{t}", 0) for t in range(T)]
    p_sell = [pulp.LpVariable(f"sell_{t}", 0) for t in range(T)]
    soc = [pulp.LpVariable(f"soc_{t}", E_min, E_max) for t in range(T)]
    u = [pulp.LpVariable(f"u_{t}", cat='Binary') for t in range(T)]

    prob += pulp.lpSum([(price_buy[t]/1000)*p_buy[t]*dt - (price_sell[t]/1000)*p_sell[t]*dt
                        for t in range(T)])

    for t in range(T):
        prob += pv[t] + p_dis[t] + p_buy[t] == load[t] + p_ch[t] + p_sell[t]
        soc_prev = SoC_init if t == 0 else soc[t-1]
        prob += soc[t] == soc_prev + eta_ch * p_ch[t] * dt - p_dis[t] * dt / eta_dis
        prob += p_ch[t] <= P_max * u[t]
        prob += p_dis[t] <= P_max * (1 - u[t])

    prob.solve(pulp.PULP_CBC_CMD(msg=0))

    res = {
        'cost': pulp.value(prob.objective),
        'p_buy': [pulp.value(p_buy[t]) for t in range(T)],
        'p_sell': [pulp.value(p_sell[t]) for t in range(T)],
        'soc': [pulp.value(soc[t]) for t in range(T)],
        'status': pulp.LpStatus[prob.status]
    }
    return res

# Solve each site independently
indep_results = []
for i in range(N_sites):
    res = solve_single_site(
        pv_profiles[i], load_profiles[i],
        price_da, price_da * 0.8,
        site_params[i][2], 0.1, site_params[i][3]
    )
    indep_results.append(res)
    print(f"Site {i}: cost = {res['cost']:.2f} EUR, status = {res['status']}")

total_indep_cost = sum(r['cost'] for r in indep_results)
print(f"\nTotal independent cost (sum): {total_indep_cost:.2f} EUR")


## 3. Centralized BRP Portfolio Optimization

Now we solve the **joint problem** where the BRP:
1. Optimizes all BESS simultaneously
2. Submits a single aggregated day-ahead nomination
3. Benefits from internal netting
4. Pays imbalance costs only on the portfolio-level deviation

### Imbalance Settlement Convention

We use the standard European convention:
- **nomination[t]** = how much BRP plans to withdraw from grid (positive = buying)
- **actual_net[t]** = actual withdrawal = portfolio_buy - portfolio_sell
- **surplus[t]** = max(nomination[t] - actual_net[t], 0) — BRP bought more on DA than needed, returns surplus
- **deficit[t]** = max(actual_net[t] - nomination[t], 0) — BRP needs more than nominated, pays premium

Settlement:
$$\text{Cost} = \sum_t \left[ \pi^{DA}_t \cdot \text{nom}_t + \pi^{def}_t \cdot \text{deficit}_t - \pi^{sur}_t \cdot \text{surplus}_t \right]$$

Where $\pi^{sur}_t < \pi^{DA}_t < \pi^{def}_t$, ensuring it is always best to nominate accurately.


In [ ]:
prob = pulp.LpProblem("BRP_Portfolio_Optimization", pulp.LpMinimize)

dt = 1.0
eta_ch = 0.95
eta_dis = 0.95

# Per-site variables
p_ch = {}
p_dis = {}
soc = {}
u_bin = {}

for i in range(N_sites):
    _, _, E_max_i, P_max_i = site_params[i]
    E_min_i = E_max_i * 0.1
    for t in range(T):
        p_ch[i,t] = pulp.LpVariable(f"ch_{i}_{t}", 0, P_max_i)
        p_dis[i,t] = pulp.LpVariable(f"dis_{i}_{t}", 0, P_max_i)
        soc[i,t] = pulp.LpVariable(f"soc_{i}_{t}", E_min_i, E_max_i)
        u_bin[i,t] = pulp.LpVariable(f"u_{i}_{t}", cat='Binary')

# Portfolio-level variables
p_portfolio_buy = [pulp.LpVariable(f"port_buy_{t}", 0) for t in range(T)]
p_portfolio_sell = [pulp.LpVariable(f"port_sell_{t}", 0) for t in range(T)]

# Day-ahead nomination (what BRP commits to withdraw from grid)
nomination = [pulp.LpVariable(f"nom_{t}", lowBound=None) for t in range(T)]

# Imbalance decomposition:
# nomination - actual_net = surplus - deficit
surplus = [pulp.LpVariable(f"surplus_{t}", 0) for t in range(T)]
deficit = [pulp.LpVariable(f"deficit_{t}", 0) for t in range(T)]

# Objective: DA cost + imbalance penalties
# DA: pay price_da for nomination (negative nomination = selling, earns money)
# Deficit: pay premium for extra energy needed
# Surplus: earn discounted price for returned energy
prob += pulp.lpSum([
    (price_da[t]/1000) * nomination[t] * dt +
    (price_imb_deficit[t]/1000) * deficit[t] * dt -
    (price_imb_surplus[t]/1000) * surplus[t] * dt
    for t in range(T)
]), "Total_BRP_Cost"

# Constraints
for t in range(T):
    for i in range(N_sites):
        _, _, E_max_i, P_max_i = site_params[i]
        SoC_init_i = E_max_i * 0.5

        # SoC dynamics
        soc_prev = SoC_init_i if t == 0 else soc[i, t-1]
        prob += soc[i,t] == soc_prev + eta_ch * p_ch[i,t] * dt - p_dis[i,t] * dt / eta_dis

        # Mutual exclusion
        prob += p_ch[i,t] <= P_max_i * u_bin[i,t]
        prob += p_dis[i,t] <= P_max_i * (1 - u_bin[i,t])

    # Portfolio power balance
    prob += (pulp.lpSum([load_profiles[i][t] + p_ch[i,t] - pv_profiles[i][t] - p_dis[i,t]
                         for i in range(N_sites)])
             == p_portfolio_buy[t] - p_portfolio_sell[t],
             f"PortfolioBalance_{t}")

    # Actual net position
    actual_net_t = p_portfolio_buy[t] - p_portfolio_sell[t]

    # Imbalance: nomination - actual = surplus - deficit
    prob += nomination[t] - actual_net_t == surplus[t] - deficit[t], f"Imbalance_{t}"

# Solve
prob.solve(pulp.PULP_CBC_CMD(msg=0))
print(f"BRP Portfolio Status: {pulp.LpStatus[prob.status]}")
print(f"BRP Portfolio Cost: {pulp.value(prob.objective):.2f} EUR")


## 4. Results Comparison


In [ ]:
# Extract results
nom_vals = [pulp.value(nomination[t]) for t in range(T)]
port_buy_vals = [pulp.value(p_portfolio_buy[t]) for t in range(T)]
port_sell_vals = [pulp.value(p_portfolio_sell[t]) for t in range(T)]
actual_vals = [port_buy_vals[t] - port_sell_vals[t] for t in range(T)]
surplus_vals = [pulp.value(surplus[t]) for t in range(T)]
deficit_vals = [pulp.value(deficit[t]) for t in range(T)]

# Independent: aggregate grid exchange
indep_agg_buy = np.array([sum(indep_results[i]['p_buy'][t] for i in range(N_sites)) for t in range(T)])
indep_agg_sell = np.array([sum(indep_results[i]['p_sell'][t] for i in range(N_sites)) for t in range(T)])
indep_net = indep_agg_buy - indep_agg_sell

fig, axes = plt.subplots(2, 2, figsize=(14, 9))

# Portfolio net position vs nomination
axes[0, 0].step(hours, nom_vals, 'b-', where='mid', linewidth=2, label='DA Nomination')
axes[0, 0].step(hours, actual_vals, 'r--', where='mid', linewidth=2, label='Actual net')
axes[0, 0].set_xlabel('Hour'); axes[0, 0].set_ylabel('kW')
axes[0, 0].set_title('BRP: DA Nomination vs Actual Net Position')
axes[0, 0].legend(); axes[0, 0].axhline(0, color='black', linewidth=0.5)

# Net grid exchange: independent vs coordinated
axes[0, 1].step(hours, indep_net, 'r--', where='mid', linewidth=2, label='Independent aggregate')
axes[0, 1].step(hours, actual_vals, 'b-', where='mid', linewidth=2, label='BRP portfolio')
axes[0, 1].set_xlabel('Hour'); axes[0, 1].set_ylabel('kW')
axes[0, 1].set_title('Net Grid Position: Independent vs BRP')
axes[0, 1].legend(); axes[0, 1].axhline(0, color='black', linewidth=0.5)

# Imbalance
axes[1, 0].bar(hours, surplus_vals, color='teal', alpha=0.7, label='Surplus')
axes[1, 0].bar(hours, [-v for v in deficit_vals], color='coral', alpha=0.7, label='Deficit')
axes[1, 0].set_xlabel('Hour'); axes[1, 0].set_ylabel('kW')
axes[1, 0].set_title('Portfolio Imbalance (Nomination - Actual)'); axes[1, 0].legend()

# SoC for all sites
for i in range(N_sites):
    soc_i = [pulp.value(soc[i,t]) for t in range(T)]
    axes[1, 1].plot(hours, soc_i, '-o', markersize=2, label=f'Site {i}')
axes[1, 1].set_xlabel('Hour'); axes[1, 1].set_ylabel('kWh')
axes[1, 1].set_title('Battery SoC (All Sites)'); axes[1, 1].legend(fontsize=7)

plt.tight_layout()
plt.savefig('/tmp/nb2_results.png', dpi=100)
plt.show()

# Cost comparison
brp_cost = pulp.value(prob.objective)
benefit = total_indep_cost - brp_cost
print(f"\n{'='*55}")
print(f"Independent (sum of single-site):  {total_indep_cost:.2f} EUR")
print(f"Centralized BRP portfolio:         {brp_cost:.2f} EUR")
print(f"Coordination benefit:              {benefit:.2f} EUR")
if abs(total_indep_cost) > 1e-6:
    print(f"Relative improvement:              {100*benefit/abs(total_indep_cost):.1f}%")
print(f"{'='*55}")


## 5. Netting Effect Analysis

Let's quantify the internal netting effect — how much portfolio aggregation reduces the total grid exchange.


In [ ]:
# Compare gross vs net grid exchange
indep_gross_buy = np.sum(indep_agg_buy)
indep_gross_sell = np.sum(indep_agg_sell)
portfolio_gross_buy = np.sum(port_buy_vals)
portfolio_gross_sell = np.sum(port_sell_vals)

print("Grid Exchange Comparison (daily totals in kWh):")
print(f"  Independent gross buy:  {indep_gross_buy:.1f}")
print(f"  Portfolio gross buy:    {portfolio_gross_buy:.1f}")
print(f"  Reduction in buying:    {indep_gross_buy - portfolio_gross_buy:.1f} "
      f"({100*(indep_gross_buy - portfolio_gross_buy)/max(indep_gross_buy,0.01):.1f}%)")
print(f"  Independent gross sell: {indep_gross_sell:.1f}")
print(f"  Portfolio gross sell:   {portfolio_gross_sell:.1f}")
print(f"  Reduction in selling:   {indep_gross_sell - portfolio_gross_sell:.1f} "
      f"({100*(indep_gross_sell - portfolio_gross_sell)/max(indep_gross_sell,0.01):.1f}%)")
print(f"\nTotal imbalance volume: {sum(surplus_vals) + sum(deficit_vals):.2f} kWh")


## 6. Key Insights for the Thesis

1. **Internal netting** is the primary value driver: when one site exports and another imports at the same hour, the BRP only pays for the net difference.

2. **Joint BESS coordination** allows batteries to be dispatched in a complementary fashion — some charge when the portfolio has surplus, others discharge during portfolio deficit.

3. **Day-ahead nomination** can be optimized jointly, reducing imbalance exposure compared to summing individual nominations.

4. **Scalability**: this centralized MILP grows linearly with the number of sites ($N \times T$ continuous variables + $N \times T$ binary variables). For 100+ sites, decomposition methods (see Notebook 4: ADMM) become necessary.

5. **BRP–Prosumer conflict**: in this centralized model the BRP has full control over all batteries. In reality, prosumers may have preferences (e.g., maintaining SoC for backup). This motivates bilevel approaches (Notebook 6).

6. **Czech market context**: OTE uses a single imbalance price derived from the system state. The dual-price model here is a simplification but captures the asymmetry between surplus and deficit. For real implementation, the actual OTE imbalance settlement rules should be used.

---

*Next: Notebook 3 adds stochastic programming to handle forecast uncertainty in PV and load.*
